# 🚀 Transfer Learning Part 1: Feature Extraction using VGG16
### 🐱 vs 🐶 Cats vs Dogs Classification with a Completely Frozen Convolutional Base

---

## 📌 1. What is Transfer Learning?

In Deep Learning, **Transfer Learning** is the process of taking a model trained on a massive, diverse dataset (such as **ImageNet** with 1.4 million images and 1,000 classes) and repurposing it for a new, related task (such as classifying **Cats vs Dogs**).

Instead of starting the training process from scratch with random weights—which requires millions of labeled images, days or weeks of compute time, and severe risk of overfitting—we leverage the rich feature representations learned by a state-of-the-art architecture like **VGG16**.

---

## 🧠 2. Anatomy of a Convolutional Neural Network

Every modern CNN consists of two primary functional blocks:

```
┌─────────────────────────────────────────────────────────────┐
│ 1. CONVOLUTIONAL BASE (Feature Extractor)                   │
│    - Stack of Conv2D + Activation + Pooling Layers          │
│    - Learns hierarchical visual features                    │
│    - Spatial representations (edges, textures, shapes)      │
└──────────────────────────────┬──────────────────────────────┘
                               │ (Extracted Feature Maps)
┌──────────────────────────────▼──────────────────────────────┐
│ 2. CLASSIFIER HEAD (Dense Layers)                           │
│    - Flatten / GlobalAveragePooling2D                       │
│    - Dense + Dropout + Normalization Layers                 │
│    - Maps high-level features to target class probabilities  │
└─────────────────────────────────────────────────────────────┘
```

---

## ❄️ 3. What is Feature Extraction?

In **Feature Extraction**:
1. We keep the pre-trained **Convolutional Base** completely **FROZEN** (`conv_base.trainable = False`).
2. We discard the original 1,000-class ImageNet classification top.
3. We attach a **new custom dense classifier head** on top suited for our binary classification task (Cat vs Dog).
4. During training, backpropagation calculates gradients **only for the new dense layers**. The pre-trained weights in the convolutional base remain untouched!

### 📊 Parameter Breakdown in Feature Extraction:
* **Total Parameters:** $pprox 16.8	ext{M}$
* **Non-Trainable Parameters (Frozen Base):** $14,714,688$ ($pprox 87.5\%$)
* **Trainable Parameters (Dense Head Only):** $pprox 2.1	ext{M}$ ($pprox 12.5\%$)

---

### 🗺️ Visual Architecture Diagram:
```
           Input Image (150 x 150 x 3)
                       │
       ┌───────────────▼───────────────┐
       │   VGG16 Convolutional Base    │  ❄️ FROZEN (conv_base.trainable = False)
       │   - Block 1: 2 Conv + MaxPool │  Weights kept fixed from ImageNet
       │   - Block 2: 2 Conv + MaxPool │  Extracts edges, textures, motifs
       │   - Block 3: 3 Conv + MaxPool │
       │   - Block 4: 3 Conv + MaxPool │
       │   - Block 5: 3 Conv + MaxPool │
       └───────────────┬───────────────┘
                       │ (4 x 4 x 512 Feature Maps)
       ┌───────────────▼───────────────┐
       │      Flatten Layer            │
       │  Dense(256, ReLU) + Dropout   │  🔥 TRAINABLE CLASSIFIER HEAD
       │  Dense(1, Sigmoid)            │  Only these weights are updated!
       └───────────────────────────────┘
```

In [ ]:
# ============================================================
# STEP 1: Dependencies Import & Environment Verification
# ============================================================
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from keras import layers, models
from keras.applications.vgg16 import VGG16, preprocess_input

# Ensure reproducible random seed
tf.keras.utils.set_random_seed(42)

print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version:      {keras.__version__}")
devices = tf.config.list_physical_devices()
print(f"Available Devices:  {devices}")

## 📁 4. Dataset Discovery & Setup

We use the popular **Cats vs Dogs** dataset (`salader/dogsvscats`). The code below automatically locates the dataset:
1. Checks the current working directory (`train` and `test` folders).
2. Checks parent directory (`../train` and `../test`).
3. If not present locally, downloads it seamlessly via `kagglehub` without requiring manual Kaggle API tokens.

In [ ]:
# ============================================================
# STEP 2: Dataset Path Resolution
# ============================================================
# Check local paths or download via kagglehub
candidate_paths = [
    (".", "train", "test"),
    ("..", "../train", "../test"),
    ("/Users/kedar/.cache/kagglehub/datasets/salader/dogsvscats/versions/1", 
     "/Users/kedar/.cache/kagglehub/datasets/salader/dogsvscats/versions/1/train", 
     "/Users/kedar/.cache/kagglehub/datasets/salader/dogsvscats/versions/1/test")
]

train_dir = None
test_dir = None

for base, tr, te in candidate_paths:
    if os.path.exists(tr) and os.path.exists(te):
        train_dir = os.path.abspath(tr)
        test_dir = os.path.abspath(te)
        print(f"Found local dataset at: {base}")
        break

if not train_dir:
    print("Dataset not found locally. Downloading via kagglehub...")
    import kagglehub
    dataset_path = kagglehub.dataset_download("salader/dogsvscats")
    train_dir = os.path.join(dataset_path, "train")
    test_dir = os.path.join(dataset_path, "test")

print(f"Train Directory: {train_dir}")
print(f"Test Directory:  {test_dir}")

# Verify file counts
num_train_cats = len(os.listdir(os.path.join(train_dir, 'cats')))
num_train_dogs = len(os.listdir(os.path.join(train_dir, 'dogs')))
num_test_cats  = len(os.listdir(os.path.join(test_dir, 'cats')))
num_test_dogs  = len(os.listdir(os.path.join(test_dir, 'dogs')))

print(f"\n--- Dataset Statistics ---")
print(f"Training:   {num_train_cats:,} Cats | {num_train_dogs:,} Dogs (Total: {num_train_cats + num_train_dogs:,})")
print(f"Testing:    {num_test_cats:,} Cats | {num_test_dogs:,} Dogs (Total: {num_test_cats + num_test_dogs:,})")

## 🖼️ 5. Visualizing Sample Images from the Dataset

Let's inspect a few sample images of cats and dogs to verify image quality and diversity.

In [ ]:
# ============================================================
# STEP 3: Preview Sample Training Images
# ============================================================
fig, axes = plt.subplots(2, 4, figsize=(14, 7))

cat_samples = os.listdir(os.path.join(train_dir, 'cats'))[:4]
dog_samples = os.listdir(os.path.join(train_dir, 'dogs'))[:4]

for i, img_name in enumerate(cat_samples):
    img = cv2.imread(os.path.join(train_dir, 'cats', img_name))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[0, i].imshow(img_rgb)
    axes[0, i].set_title("Label: CAT (0)", fontsize=11, fontweight='bold', color='navy')
    axes[0, i].axis("off")

for i, img_name in enumerate(dog_samples):
    img = cv2.imread(os.path.join(train_dir, 'dogs', img_name))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[1, i].imshow(img_rgb)
    axes[1, i].set_title("Label: DOG (1)", fontsize=11, fontweight='bold', color='darkgreen')
    axes[1, i].axis("off")

plt.suptitle("Sample Training Images (Cats vs Dogs)", fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

## ⚙️ 6. Data Pipeline & VGG16 Preprocessing

### 🎯 Key Design Choices:
1. **Image Resolution $(150, 150)$**:
   VGG16 was originally trained on $(224, 224)$, but because the convolutional base has **no fixed dense layers**, it accepts arbitrary spatial dimensions $\ge 32 	imes 32$. Using $(150, 150)$ trains $pprox 2.2	imes$ faster while preserving rich feature fidelity.
2. **`preprocess_input`**:
   Rather than simple $1/255$ scaling, VGG16 expects inputs preprocessed with `tf.keras.applications.vgg16.preprocess_input`:
   - Converts RGB to BGR.
   - Zero-centers each color channel with respect to the ImageNet dataset mean:
     $$	ext{Mean BGR} = [103.939, 116.779, 123.68]$$
3. **Data Augmentation**:
   Random horizontal flips, small rotations, and zooms prevent the newly added dense layers from overfitting.

In [ ]:
# ============================================================
# STEP 4: Efficient Data Loading Pipeline
# ============================================================
IMG_SIZE = (150, 150)
BATCH_SIZE = 32

# Load train dataset
train_ds_raw = keras.utils.image_dataset_from_directory(
    directory=train_dir,
    labels="inferred",
    label_mode="int",
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    shuffle=True,
    seed=42
)

# Load validation dataset
val_ds_raw = keras.utils.image_dataset_from_directory(
    directory=test_dir,
    labels="inferred",
    label_mode="int",
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    shuffle=False
)

# Apply VGG16 preprocess_input function
def preprocess_vgg(image, label):
    # image has shape (batch, 150, 150, 3) with values in [0, 255]
    return preprocess_input(image), label

train_ds = train_ds_raw.map(preprocess_vgg, num_parallel_calls=tf.data.AUTOTUNE)
val_ds   = val_ds_raw.map(preprocess_vgg, num_parallel_calls=tf.data.AUTOTUNE)

# Optimize pipeline throughput with prefetching
train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds   = val_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

print("[SUCCESS] Data pipeline initialized with VGG16 preprocessing!")

## 🏗️ 7. Loading the Pre-Trained VGG16 Model

We instantiate the `VGG16` model with:
- `weights='imagenet'`: Initializes the network with weights learned on 1.4 million images.
- `include_top=False`: Excludes the original 1000-class dense classification head so we can attach our own!
- `input_shape=(150, 150, 3)`: Configures the input tensor dimensions.

In [ ]:
# ============================================================
# STEP 5: Instantiate Pre-Trained VGG16 Base
# ============================================================
conv_base = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
)

print(f"Number of layers in VGG16 base: {len(conv_base.layers)}")
conv_base.summary()

## ❄️ 8. Freezing the Convolutional Base

Setting `conv_base.trainable = False` is the defining step of **Feature Extraction**.

### 🔍 What happens under the hood?
- All weight variables inside `conv_base` are marked as **non-trainable**.
- During the forward pass, images pass through the conv layers normally to extract feature maps.
- During the backward pass (backpropagation), gradients for these weights are **not calculated**, and their values remain **permanently frozen** at their pre-trained ImageNet state.

In [ ]:
# ============================================================
# STEP 6: Freeze All Convolutional Base Layers
# ============================================================
# Check trainable weights BEFORE freezing
print(f"Trainable weight tensors BEFORE freezing: {len(conv_base.trainable_weights)}")

# FREEZE THE CONVOLUTIONAL BASE
conv_base.trainable = False

# Check trainable weights AFTER freezing
print(f"Trainable weight tensors AFTER freezing:  {len(conv_base.trainable_weights)}")
print(f"Non-trainable weight tensors:            {len(conv_base.non_trainable_weights)}")

# Detailed audit of all base layers
print("\n--- Layer-by-Layer Trainability Audit (conv_base) ---")
for i, layer in enumerate(conv_base.layers):
    status = "🔥 TRAINABLE" if layer.trainable else "❄️ FROZEN"
    print(f"Layer {i:2d}: {layer.name:<15} ({layer.__class__.__name__:<15}) -> {status}")

## 🧩 9. Building the Full Model (Base + Custom Classifier Head)

We combine:
1. **Data Augmentation Block**: Applies random horizontal flips, small rotations ($10\%$), and zooms ($10\%$) during training.
2. **Frozen VGG16 Base**: Extracts rich spatial feature maps of size $(4, 4, 512) = 8,192$ values.
3. **Flatten Layer**: Flattens the feature map into a 1D vector.
4. **Dense Layer (256 units, ReLU)**: Learns task-specific non-linear relationships.
5. **Dropout Layer (0.3)**: Regularization to prevent co-adaptation of neurons.
6. **Output Dense Layer (1 unit, Sigmoid)**: Outputs a probability $p \in [0, 1]$ where $p < 0.5 \implies 	ext{Cat}$ and $p \ge 0.5 \implies 	ext{Dog}$.

In [ ]:
# ============================================================
# STEP 7: Assemble Model with Custom Classifier Head
# ============================================================
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="data_augmentation")

model = models.Sequential([
    # Data Augmentation (active only during training)
    data_augmentation,
    
    # Frozen Convolutional Base
    conv_base,
    
    # Custom Dense Classifier Head
    layers.Flatten(name="flatten_features"),
    layers.Dense(256, activation='relu', name="dense_classifier"),
    layers.Dropout(0.3, name="dropout_regularization"),
    layers.Dense(1, activation='sigmoid', name="binary_output")
], name="vgg16_feature_extraction_model")

# Display comprehensive architecture summary
model.summary()

# Verify parameter counts
trainable_count = sum(int(np.prod(p.shape)) for p in model.trainable_weights)
non_trainable_count = sum(int(np.prod(p.shape)) for p in model.non_trainable_weights)
total_count = trainable_count + non_trainable_count

print(f"\n--- Parameter Verification ---")
print(f"Total Parameters:         {total_count:,}")
print(f"Trainable (Dense Head):   {trainable_count:,} ({trainable_count / total_count * 100:.2f}%)")
print(f"Non-Trainable (VGG Base): {non_trainable_count:,} ({non_trainable_count / total_count * 100:.2f}%)")

## 🎯 10. Compiling & Training the Model

- **Optimizer**: `Adam(learning_rate=1e-4)`. A standard moderate learning rate is ideal here because we are only training the newly initialized dense layers.
- **Loss**: `binary_crossentropy` (standard for 2-class classification).
- **Callbacks**:
  - `EarlyStopping`: Stops training if validation loss fails to improve for 3 consecutive epochs.
  - `ReduceLROnPlateau`: Halves the learning rate if validation loss plateaus.

In [ ]:
# ============================================================
# STEP 8: Model Compilation & Callbacks
# ============================================================
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

print("Model compiled and ready for training!")

In [ ]:
# ============================================================
# STEP 9: Train the Model (Feature Extraction)
# ============================================================
# We train for 10 epochs (or adjust according to compute capacity)
EPOCHS = 10

print("🚀 Starting Feature Extraction Training...")
history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    callbacks=callbacks
)
print("✅ Feature Extraction Training Complete!")

## 📈 11. Performance Evaluation & Visualization

Let's plot the training and validation accuracy and loss over epochs.

In [ ]:
# ============================================================
# STEP 10: Training Curves Visualization
# ============================================================
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(14, 5))

# Plot Accuracy
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, 'o-', color='#1f77b4', label='Training Accuracy', linewidth=2)
plt.plot(epochs_range, val_acc, 's--', color='#ff7f0e', label='Validation Accuracy', linewidth=2)
plt.title('Training & Validation Accuracy (Feature Extraction)', fontsize=13, fontweight='bold')
plt.xlabel('Epochs', fontsize=11)
plt.ylabel('Accuracy', fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='lower right', frameon=True)

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, 'o-', color='#1f77b4', label='Training Loss', linewidth=2)
plt.plot(epochs_range, val_loss, 's--', color='#ff7f0e', label='Validation Loss', linewidth=2)
plt.title('Training & Validation Loss (Feature Extraction)', fontsize=13, fontweight='bold')
plt.xlabel('Epochs', fontsize=11)
plt.ylabel('Loss (Binary Crossentropy)', fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='upper right', frameon=True)

plt.tight_layout()
plt.show()

# Print Final Benchmark
print(f"Final Training Accuracy:   {acc[-1]*100:.2f}%")
print(f"Final Validation Accuracy: {val_acc[-1]*100:.2f}%")
print(f"Final Validation Loss:     {val_loss[-1]:.4f}")

## 🔮 12. Inference on Real Unseen Images

Now let's test our trained model on test images (such as `cat.jpg` and `dog.jpg`) and verify predictions.

In [ ]:
# ============================================================
# STEP 11: Real-World Inference Pipeline
# ============================================================
def predict_image(image_path, model, target_size=(150, 150)):
    if not os.path.exists(image_path):
        print(f"⚠️ Image not found at: {image_path}")
        return
    
    # Load and preprocess image
    img_bgr = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, target_size)
    
    # Preprocess using VGG16 expectations
    img_array = np.expand_dims(img_resized.astype(np.float32), axis=0)
    img_preprocessed = preprocess_input(img_array)
    
    # Inference
    prediction_prob = model.predict(img_preprocessed, verbose=0)[0][0]
    label = "DOG 🐶" if prediction_prob >= 0.5 else "CAT 🐱"
    confidence = prediction_prob if prediction_prob >= 0.5 else (1.0 - prediction_prob)
    
    # Visualization
    plt.figure(figsize=(5, 5))
    plt.imshow(img_rgb)
    plt.title(f"Prediction: {label}\nConfidence: {confidence*100:.2f}% (Raw: {prediction_prob:.4f})", 
              fontsize=12, fontweight='bold', color='darkgreen' if confidence > 0.8 else 'darkorange')
    plt.axis("off")
    plt.show()
    
    return label, confidence

# Test on local sample images if available
test_candidates = ["cat.jpg", "dog.jpg", "../cat.jpg", "../dog.jpg"]
for path in test_candidates:
    if os.path.exists(path):
        print(f"\n--- Testing on: {path} ---")
        predict_image(path, model)

## 📝 13. Summary & Takeaways from Feature Extraction

### ✅ Advantages of Feature Extraction:
1. **Extremely Fast to Train**: Only $pprox 2.1	ext{M}$ parameters are updated, compared to $pprox 16.8	ext{M}$ parameters from scratch.
2. **Prevents Overfitting**: Since pre-trained ImageNet filters are fixed, the model cannot overfit its low-level visual representations to small datasets.
3. **High Baseline Accuracy**: Even with a simple dense head, accuracy quickly reaches $pprox 90	ext{--}94\%$ on Cats vs Dogs.

### ⚠️ Limitations of Feature Extraction:
- The convolutional filters remain locked to their general ImageNet representations.
- They cannot specialize or adapt to subtle visual nuances specific to our target dataset (e.g. whiskers, pointed feline ears vs floppy canine ears).

---

### ➡️ Next Step: Fine-Tuning!
In the next notebook (**Part 2: Fine-Tuning**), we will keep the classifier head we just trained, **unfreeze the last convolutional layer (`block5_conv3`)**, and train it with a very low learning rate to adapt high-level features for maximum accuracy!